# Cài đặt thư viện và Import

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
!pip install transformers datasets scikit-learn

import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, set_seed
from sklearn.metrics import accuracy_score, f1_score

# Đọc dữ liệu và Ánh xạ nhãn

In [14]:
# 1. Đọc dữ liệu

train_df = pd.read_csv('/content/train.csv')
val_df = pd.read_csv('/content/val.csv')
test_df = pd.read_csv('/content/test.csv')

# 2. Map nhãn chuẩn: fake=0, real=1
label_map = {'fake': 0, 'real': 1}
for df in [train_df, val_df, test_df]:
    df['label'] = df['label'].map(label_map)

# 3. Chuyển đổi sang format Dataset của Hugging Face
train_dataset = Dataset.from_pandas(train_df[['tweet', 'label']])
val_dataset = Dataset.from_pandas(val_df[['tweet', 'label']])
test_dataset = Dataset.from_pandas(test_df[['tweet', 'label']])

# Tokenizer(Tiền xử lý văn bản)

In [15]:
# Tải tokenizer của BERT-base
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    # Padding và cắt gọn văn bản (max_length có thể tinh chỉnh, ví dụ 128)
    return tokenizer(examples['tweet'], padding='max_length', truncation=True, max_length=128)

# Áp dụng tokenizer lên toàn bộ tập dữ liệu
train_tokenized = train_dataset.map(tokenize_function, batched=True)
val_tokenized = val_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/6420 [00:00<?, ? examples/s]

Map:   0%|          | 0/2140 [00:00<?, ? examples/s]

Map:   0%|          | 0/2140 [00:00<?, ? examples/s]

# Thiết lập đánh giá và Vòng lặp huấn luyện với 3 Seed

In [16]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    # Trả về các metric trên thang 0-1
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1_weighted': f1_score(labels, predictions, average='weighted')
    }

# Yêu cầu 3 seed
seeds = [42, 123, 2024]

for seed in seeds:
    print(f"\n{'='*40}")
    print(f"BẮT ĐẦU HUẤN LUYỆN VỚI SEED: {seed}")
    print(f"{'='*40}")

    # Cố định seed cho khả năng tái tạo (reproducibility)
    set_seed(seed)

    # Khởi tạo mô hình MỚI cho mỗi lần lặp seed
    model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

    # Tạo thư mục xuất kết quả riêng rẽ cho từng seed
    output_dir = f'./bert_base_seed_{seed}'

    # Cấu hình Training
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,                     # Số vòng lặp huấn luyện, có thể tăng lên 4 hoặc 5 tùy bạn
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        eval_strategy="epoch",            # Đánh giá sau mỗi epoch
        save_strategy="epoch",                  # Lưu checkpoint sau mỗi epoch
        metric_for_best_model="f1_weighted",    # TIÊU CHÍ QUAN TRỌNG: Chọn checkpoint tốt nhất bằng f1_weighted
        load_best_model_at_end=True,            # Tự động nạp lại checkpoint tốt nhất vào cuối quá trình train
        seed=seed
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=val_tokenized,
        compute_metrics=compute_metrics
    )

    # Bắt đầu train
    trainer.train()

    # Đánh giá lại trên tập Validation để ghi log
    val_results = trainer.evaluate()
    print(f"\nKết quả trên tập Validation (Seed {seed}):")
    print(val_results)


BẮT ĐẦU HUẤN LUYỆN VỚI SEED: 42


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,No log,0.102262,0.970093,0.970070
2,0.182806,0.113043,0.978037,0.978036
3,0.041642,0.112923,0.979439,0.979432


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Accuracy,F1 Weighted
0.036536,0.092020,3,0.980374,0.980368



Kết quả trên tập Validation (Seed 2024):
{'eval_loss': 0.09201952815055847, 'eval_accuracy': 0.9803738317757009, 'eval_f1_weighted': 0.9803680561578646}


# Error Analysis (Phân tích lỗi)

In [17]:
import numpy as np

print("Đang tiến hành phân tích lỗi trên tập Validation...")
# Dự đoán đồng loạt bằng trainer (tối ưu hơn pipeline rất nhiều)
predictions, labels, _ = trainer.predict(val_tokenized)
preds = np.argmax(predictions, axis=1)

# Lọc ra các vị trí dự đoán sai
error_indices = np.where(preds != labels)[0]
print(f"Tổng số câu dự đoán sai: {len(error_indices)}\n")

# In ra phân tích 10 mẫu đầu tiên
for idx, i in enumerate(error_indices[:10]):
    tweet = val_dataset[int(i)]['tweet']
    true_label = val_dataset[int(i)]['label']
    pred_label = preds[i]

    true_text = "Real (1)" if true_label == 1 else "Fake (0)"
    pred_text = "Real (1)" if pred_label == 1 else "Fake (0)"

    print(f"--- Mẫu sai số {idx + 1} ---")
    print(f"Thực tế: {true_text} | Mô hình đoán: {pred_text}")
    print(f"Nội dung: {tweet}\n")

Đang tiến hành phân tích lỗi trên tập Validation...


Tổng số câu dự đoán sai: 42

--- Mẫu sai số 1 ---
Thực tế: Real (1) | Mô hình đoán: Fake (0)
Nội dung: Tomorrow April 6 we will pass 10000 coronavirus deaths. We passed 5000 on April 2. We passed 1000 on March 26. We passed 100 on March 18.

--- Mẫu sai số 2 ---
Thực tế: Fake (0) | Mô hình đoán: Real (1)
Nội dung: The Global #dietarysupplements market is anticipated to grow at a #CAGR of around 7.50% during 2020-25. 
Check the full report:- https://t.co/71J6LqCvgr

#food #beverage #marketresearch #marketstrategy #USA #animalhusbandry #covid_19 #kerrygroup #nutrients #Immunity #Energy https://t.co/XPFXA1zzo7

--- Mẫu sai số 3 ---
Thực tế: Fake (0) | Mô hình đoán: Real (1)
Nội dung: There is no evidence that children have died because of a COVID-19 vaccine. No vaccine currently in development has been approved for widespread public use. https://t.co/9ecvMR8SAf

--- Mẫu sai số 4 ---
Thực tế: Real (1) | Mô hình đoán: Fake (0)
Nội dung: What will remain of at-home and telehealth innovations

# ĐÁNH GIÁ TRÊN TẬP TEST CHO 3 SEED

In [21]:

import pandas as pd
from transformers.trainer_utils import get_last_checkpoint

print("="*50)
print("TIẾN HÀNH ĐÁNH GIÁ TRÊN TẬP TEST (test.csv)")
print("="*50)

test_results = []

for seed in seeds:
    output_dir = f'./bert_base_seed_{seed}'

    checkpoint_dir = get_last_checkpoint(output_dir)
    if checkpoint_dir is None:
        checkpoint_dir = output_dir

    print(f"Đang nạp mô hình (Seed {seed}) từ: {checkpoint_dir}")

    # Nạp lại mô hình tốt nhất
    model_test = BertForSequenceClassification.from_pretrained(checkpoint_dir, num_labels=2)
    model_test.to('cuda' if torch.cuda.is_available() else 'cpu')

    # Khởi tạo Trainer chỉ để đánh giá
    trainer_test = Trainer(
        model=model_test,
        compute_metrics=compute_metrics
    )

    # Đánh giá trên test_tokenized
    res = trainer_test.evaluate(test_tokenized)
    print(f"--> Kết quả Test (Seed {seed}): Accuracy: {res['eval_accuracy']:.4f} | F1: {res['eval_f1_weighted']:.4f}\n")

    test_results.append({
        'Seed': seed,
        'Accuracy': res['eval_accuracy'],
        'F1 Weighted': res['eval_f1_weighted']
    })

# In bảng tổng hợp
df_test = pd.DataFrame(test_results)
print("\nBẢNG TỔNG HỢP KẾT QUẢ TEST:")
print(df_test)
print(f"\n=> F1 Weighted Trung bình: {df_test['F1 Weighted'].mean():.6f} ± {df_test['F1 Weighted'].std():.6f}")

TIẾN HÀNH ĐÁNH GIÁ TRÊN TẬP TEST (test.csv)
Đang nạp mô hình (Seed 42) từ: ./bert_base_seed_42/checkpoint-1206


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Training Loss,Validation Loss,Step,Accuracy,F1 Weighted
No log,0.100170,0,0.981308,0.981304


--> Kết quả Test (Seed 42): Accuracy: 0.9813 | F1: 0.9813

Đang nạp mô hình (Seed 123) từ: ./bert_base_seed_123/checkpoint-1206


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Training Loss,Validation Loss,Step,Accuracy,F1 Weighted
No log,0.111971,0,0.979439,0.979436


--> Kết quả Test (Seed 123): Accuracy: 0.9794 | F1: 0.9794

Đang nạp mô hình (Seed 2024) từ: ./bert_base_seed_2024/checkpoint-1206


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Training Loss,Validation Loss,Step,Accuracy,F1 Weighted
No log,0.105432,0,0.979439,0.979437


--> Kết quả Test (Seed 2024): Accuracy: 0.9794 | F1: 0.9794


BẢNG TỔNG HỢP KẾT QUẢ TEST:
   Seed  Accuracy  F1 Weighted
0    42  0.981308     0.981304
1   123  0.979439     0.979436
2  2024  0.979439     0.979437

=> F1 Weighted Trung bình: 0.980059 ± 0.001078


# THỬ NGHIỆM VÀ PHÂN TÍCH CÂU THỰC TẾ TRÊN MẠNG

In [23]:

from transformers.trainer_utils import get_last_checkpoint

print("\n" + "="*50)
print("THỬ NGHIỆM MÔ HÌNH VỚI DỮ LIỆU THỰC TẾ TỪ INTERNET")
print("="*50)

# Tự động tìm thư mục checkpoint của seed 2024 (seed tốt nhất)
best_seed_dir = './bert_base_seed_2024'
best_checkpoint = get_last_checkpoint(best_seed_dir)
if best_checkpoint is None:
    best_checkpoint = best_seed_dir

print(f"Đang nạp mô hình từ: {best_checkpoint}")

# Nạp mô hình
best_model = BertForSequenceClassification.from_pretrained(best_checkpoint)
best_model.to('cuda' if torch.cuda.is_available() else 'cpu')

# Bộ dữ liệu 7 câu thực tế ngoài tập Train/Val/Test
internet_samples = [
    {
        "text": "According to the CDC, COVID-19 vaccines are safe and effective. Millions of people have received COVID-19 vaccines under the most intense safety monitoring in U.S. history.",
        "source": "CDC Official Website",
        "true_label": 1 # Real
    },
    {
        "text": "The WHO states that antibiotics do not work against viruses, they only work on bacterial infections. Antibiotics should not be used to treat COVID-19.",
        "source": "WHO Mythbusters",
        "true_label": 1 # Real
    },
    {
        "text": "Holding your breath for 10 seconds or more without coughing or feeling discomfort means you are free from the coronavirus disease.",
        "source": "Internet Myth (Bác bỏ bởi WHO)",
        "true_label": 0 # Fake
    },
    {
        "text": "COVID-19 vaccines contain microchips to track people's movements and control populations via 5G towers.",
        "source": "Conspiracy Theory / Reuters Fact Check",
        "true_label": 0 # Fake
    },
    {
        "text": "Drinking alcohol or spraying chlorine all over your body will kill the new coronavirus.",
        "source": "Internet Rumor",
        "true_label": 0 # Fake
    },
    # 2 CÂU ĐẶC BIỆT MÀ BÁO CÁO CỦA NHÓM ĐÃ CHỈ RA LỖI TỪ DISTILBERT
    {
        "text": "People of all ages can get infected by the new coronavirus.",
        "source": "WHO Viet Nam",
        "true_label": 1 # Real
    },
    {
        "text": "No, drinking alcohol does not protect you from COVID-19 infection.",
        "source": "WHO Viet Nam",
        "true_label": 1 # Real
    }
]

def test_wild_samples(samples, model, tokenizer):
    model.eval()
    wrong_preds = []

    for i, sample in enumerate(samples):
        text = sample["text"]
        true_lbl = sample["true_label"]
        src = sample["source"]

        inputs = tokenizer(text, padding='max_length', truncation=True, max_length=128, return_tensors="pt")
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            pred_lbl = torch.argmax(outputs.logits, axis=-1).item()

        true_name = "Real (1)" if true_lbl == 1 else "Fake (0)"
        pred_name = "Real (1)" if pred_lbl == 1 else "Fake (0)"
        status = "✅ ĐÚNG" if true_lbl == pred_lbl else "❌ SAI"

        print(f"Mẫu {i+1} [{status}]: {text}")
        print(f"Nguồn: {src} | Nhãn thực tế: {true_name} | Dự đoán: {pred_name}\n")

        if true_lbl != pred_lbl:
            wrong_preds.append((sample, pred_name))

    return wrong_preds

wrong_predictions = test_wild_samples(internet_samples, best_model, tokenizer)

# -- TỔNG KẾT PHÂN TÍCH LỖI --
print("-" * 50)
print("PHÂN TÍCH LỖI TRÊN CÁC CÂU NGOÀI DỮ LIỆU:")
if len(wrong_predictions) == 0:
    print("Mô hình dự đoán đúng 100% các câu thực tế ngoài dữ liệu.")
else:
    for sample, pred in wrong_predictions:
        print(f"-> Text: {sample['text']}")
        print(f"   Lỗi: Mô hình đoán {pred} trong khi thực tế là {'Real' if sample['true_label']==1 else 'Fake'}.")
        print("   Phân tích: Câu này lấy từ nguồn thực tế nhưng không chứa các dấu hiệu quen thuộc trong tập Train (như link URL t.co, hashtag). Việc thiếu metadata hoặc sử dụng cấu trúc phủ định ('No', 'does not') khiến mô hình bị nhầm lẫn giữa phong cách văn bản và độ xác thực thông tin.\n")


THỬ NGHIỆM MÔ HÌNH VỚI DỮ LIỆU THỰC TẾ TỪ INTERNET
Đang nạp mô hình từ: ./bert_base_seed_2024/checkpoint-1206


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Mẫu 1 [❌ SAI]: According to the CDC, COVID-19 vaccines are safe and effective. Millions of people have received COVID-19 vaccines under the most intense safety monitoring in U.S. history.
Nguồn: CDC Official Website | Nhãn thực tế: Real (1) | Dự đoán: Fake (0)

Mẫu 2 [❌ SAI]: The WHO states that antibiotics do not work against viruses, they only work on bacterial infections. Antibiotics should not be used to treat COVID-19.
Nguồn: WHO Mythbusters | Nhãn thực tế: Real (1) | Dự đoán: Fake (0)

Mẫu 3 [✅ ĐÚNG]: Holding your breath for 10 seconds or more without coughing or feeling discomfort means you are free from the coronavirus disease.
Nguồn: Internet Myth (Bác bỏ bởi WHO) | Nhãn thực tế: Fake (0) | Dự đoán: Fake (0)

Mẫu 4 [✅ ĐÚNG]: COVID-19 vaccines contain microchips to track people's movements and control populations via 5G towers.
Nguồn: Conspiracy Theory / Reuters Fact Check | Nhãn thực tế: Fake (0) | Dự đoán: Fake (0)

Mẫu 5 [✅ ĐÚNG]: Drinking alcohol or spraying chlorine all ov

In [24]:
!zip -r config_seed_2024.zip ./bert_base_seed_2024 -x "*model.safetensors"

  adding: bert_base_seed_2024/ (stored 0%)
  adding: bert_base_seed_2024/checkpoint-1206/ (stored 0%)
  adding: bert_base_seed_2024/checkpoint-1206/trainer_state.json (deflated 64%)
  adding: bert_base_seed_2024/checkpoint-1206/rng_state.pth (deflated 26%)
  adding: bert_base_seed_2024/checkpoint-1206/config.json (deflated 53%)
  adding: bert_base_seed_2024/checkpoint-1206/scheduler.pt (deflated 61%)
  adding: bert_base_seed_2024/checkpoint-1206/optimizer.pt (deflated 20%)
  adding: bert_base_seed_2024/checkpoint-1206/training_args.bin (deflated 53%)
  adding: bert_base_seed_2024/checkpoint-804/ (stored 0%)
  adding: bert_base_seed_2024/checkpoint-804/trainer_state.json (deflated 60%)
  adding: bert_base_seed_2024/checkpoint-804/rng_state.pth (deflated 26%)
  adding: bert_base_seed_2024/checkpoint-804/config.json (deflated 53%)
  adding: bert_base_seed_2024/checkpoint-804/scheduler.pt (deflated 61%)
  adding: bert_base_seed_2024/checkpoint-804/optimizer.pt (deflated 20%)
  adding: bert

In [18]:
# ---DEMO INFERENCE---
def predict_fake_news(text):
    print(f"Đang phân tích câu: '{text}'")
    # Tiền xử lý câu văn nhập vào
    inputs = tokenizer(text, padding='max_length', truncation=True, max_length=128, return_tensors="pt")

    # Đẩy dữ liệu vào cùng thiết bị với mô hình (GPU/CPU)
    inputs = {k: v.to(trainer.model.device) for k, v in inputs.items()}

    # Dự đoán
    trainer.model.eval()
    with torch.no_grad():
        outputs = trainer.model(**inputs)

    # Lấy nhãn có xác suất cao nhất
    prediction = torch.argmax(outputs.logits, axis=-1).item()
    label_name = "Real (Tin thật)" if prediction == 1 else "Fake (Tin giả)"
    print(f"=> Kết quả dự đoán: {label_name}\n")

cau_thu_nghiem = "States reported 1121 deaths a small rise from last Tuesday. Southern states reported 640 of those deaths. https://t.co/YASGRTT4ux,"
predict_fake_news(cau_thu_nghiem)

Đang phân tích câu: 'States reported 1121 deaths a small rise from last Tuesday. Southern states reported 640 of those deaths. https://t.co/YASGRTT4ux,'
=> Kết quả dự đoán: Real (Tin thật)

